# Suivi d'expériences avec MLflow : Iris + RandomForest

Exécutez les cellules **dans l'ordre** (Maj + Entrée).

## 1. Imports

In [1]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split

## 2. Configuration de MLflow

- `set_tracking_uri` : où MLflow enregistre les résultats (une base SQLite locale `mlflow.db`).
- `set_experiment` : le nom de l'expérience qui regroupe tous nos essais.

In [2]:
# client.delete_experiment("1")
# client.delete_registered_model("iris-classifier")

In [3]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("iris-classification")

2026/10/18 22:05:37 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/18 22:05:37 INFO mlflow.store.db.utils: Updating database tables
2026/10/18 22:05:38 INFO mlflow.tracking.fluent: Experiment with name 'iris-classification' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/bertrandfillion/Python Projects/mlflow/mlflow-iris/mlruns/1', creation_time=1792353938303, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792353938303, lifecycle_stage='active', name='iris-classification', tags={}, trace_location=None, workspace='default'>

## 3. Préparation des données

In [4]:
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Entraînement : {len(X_train)} exemples, test : {len(X_test)} exemples")

Entraînement : 120 exemples, test : 30 exemples


## 4. Une fonction qui entraîne un modèle et l'enregistre

Chaque appel crée un **run** MLflow. Dans ce run, on enregistre :
- les **paramètres** (réglages du modèle),
- les **métriques** (résultats),
- le **modèle** lui-même (artefact).

Le paramètre `skops_trusted_types` déclare comme sûr un objet interne des forêts aléatoires (`sklearn.tree._tree.Tree`). MLflow l'exige car il enregistre les modèles dans un format sécurisé ; ici, le modèle est créé par nous-mêmes.

In [5]:
def train_model(model_class, params, run_name=None):
    with mlflow.start_run(run_name=run_name):
        model = model_class(**params)
        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)
        f1 = f1_score(y_test, predictions, average="macro")

        mlflow.log_param("model_type", model_class.__name__)
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_macro", f1)
        mlflow.sklearn.log_model(
            model,
            name="model",
            skops_trusted_types=["sklearn.tree._tree.Tree"],
        )

        print(f"{model_class.__name__} {params} -> accuracy={accuracy:.3f}, f1={f1:.3f}")

## 5. Premier essai

In [6]:
train_model(RandomForestClassifier, {"n_estimators": 10, "max_depth": 2, "random_state": 42}, run_name="rf-essai")
train_model(LogisticRegression, {"C": 0.001, "max_iter": 200}, run_name="logreg-bridee")

2026/10/18 22:05:38 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/bertrandfillion/Python Projects/mlflow/mlflow-iris
2026/10/18 22:05:38 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/bertrandfillion/Python Projects/mlflow/mlflow-iris
2026/10/18 22:05:38 INFO mlflow.utils.environment: Detected uv project at /Users/bertrandfillion/Python Projects/mlflow/mlflow-iris. Attempting to export requirements via 'uv export'.
2026/10/18 22:05:38 INFO mlflow.utils.uv_utils: Exported 90 dependencies via uv
2026/10/18 22:05:38 INFO mlflow.utils.environment: Successfully exported 90 requirements from uv project. Skipping package capture based inference.
2026/10/18 22:05:39 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/10/18 22:05:39 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock an

RandomForestClassifier {'n_estimators': 10, 'max_depth': 2, 'random_state': 42} -> accuracy=1.000, f1=1.000
LogisticRegression {'C': 0.001, 'max_iter': 200} -> accuracy=0.933, f1=0.931


## 6. Meilleur modèle
On recherche dans les runs celui qui a la meilleure valeur pour la métrique accuracy

### Récupération des runs

In [7]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")

runs = mlflow.search_runs(
    experiment_names=["iris-classification"],
    order_by=["metrics.accuracy DESC"],
    max_results=1,
)

print(runs[["run_id", "metrics.accuracy", "params.model_type"]])

                             run_id  metrics.accuracy       params.model_type
0  bee725d41e524e62ae565f4ef6f5c240               1.0  RandomForestClassifier


In [8]:
best_run_id = runs.loc[0, "run_id"] # on a trié par ordre décroissant, donc le premier est le meilleur

version = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name="iris-classifier",
)
print(version.name, version.version)

Successfully registered model 'iris-classifier'.
2026/10/18 22:05:39 WARNING mlflow.tracking._model_registry.fluent: Run with id bee725d41e524e62ae565f4ef6f5c240 has no artifacts at artifact path 'model', registering model based on models:/m-1b905ca464d34578923fcdf92efdff22 instead


Created version '1' of model 'iris-classifier'.


iris-classifier 1


### Créer le champion
On crée un alias "champion" pour le meilleur run

In [9]:
from mlflow import MlflowClient

client = MlflowClient()
client.set_registered_model_alias("iris-classifier", "champion", version.version)

champion = client.get_model_version_by_alias("iris-classifier", "champion")
print(champion.name, champion.version)

iris-classifier 1


### Retrouver le champion

In [10]:
champion = client.get_model_version_by_alias("iris-classifier", "champion")
champion_run = client.get_run(champion.run_id)
champion_accuracy = champion_run.data.metrics["accuracy"]

print(champion.run_id, champion_accuracy)

bee725d41e524e62ae565f4ef6f5c240 1.0


In [11]:
all_runs = mlflow.search_runs(
    experiment_names=["iris-classification"],
    order_by=["metrics.accuracy DESC"],
)
print(all_runs[["run_id", "tags.mlflow.runName", "metrics.accuracy", "params.model_type"]])

                             run_id tags.mlflow.runName  metrics.accuracy  \
0  bee725d41e524e62ae565f4ef6f5c240            rf-essai          1.000000   
1  1009a64356754532a21609bdf19dc221       logreg-bridee          0.933333   

        params.model_type  
0  RandomForestClassifier  
1      LogisticRegression  


## 7. MLFOW UI

Ouvrez un terminal dans le dossier du projet et lancez :

```bash
uv run mlflow ui --backend-store-uri sqlite:///mlflow.db
```

Puis visitez http://127.0.0.1:5000

## Debug

### Suppression de modèle

In [ ]:
# client.delete_model_version("iris-classifier", "1")

In [17]:
for v in client.search_model_versions("name='iris-classifier'"):
    print(v.version, v.aliases)
    